In [1]:
from astropy.table import Table
from astropy.coordinates import SkyCoord
from astropy import units as u, constants as c
import pandas as pd

In [2]:
psr_names = pd.read_csv('paper_df_postprocessed.csv')['JNAME']

In [3]:
container = pd.read_csv('cand_controls_bypsr_postprocessed/J1809-1943.csv')
container.drop(container.index, inplace=True)

In [5]:
for name in psr_names:
    data = pd.read_csv('cand_controls_bypsr/' + name + '.csv')
    coords = SkyCoord(
        ra=data["ra_deg_cont"] * u.degree, dec=data["dec_deg_cont"] * u.degree
    )

    # Find the nearest neighbour
    _, sep, _ = coords.match_to_catalog_sky(coords, nthneighbor=2)


    # Demand that nearest neighbour is at least 10" away
    pos_mask = sep.arcsec >= 10

    # Select bright sources (SNR >=8)
    snr = data["flux_peak"] / data["rms_image"]

    snr_mask = snr >= 8

    # Combine the masks
    mask = (pos_mask) & (snr_mask)

    # Filter the data
    cut_data = data[mask]
    
    container = pd.concat([container, cut_data])


In [1]:
container['psr_name']

NameError: name 'container' is not defined

In [7]:
container.to_csv('test_cand_controls_truncated_UPDATED_postprocessed.csv')